# Laboratorio L9 - Quali segnali restano utili

B.4 - Cybersicurezza e IA

Un semplice controllo a regole viene applicato a due gruppi di messaggi fraudolenti inventati per il corso:

- gruppo A: messaggi "vecchio stile", con errori e formattazione trascurata
- gruppo B: messaggi con testo curato, come quelli che produce un modello linguistico

Domanda di fondo: quali regole riconoscono entrambi i gruppi?

## 1. I messaggi

In [ ]:
GRUPPO_A = [
    "URGENTE!!! Gentile cliente il suo conto e stato BLOCCATO per motivi di sicureza. Clicca qui immediatamente per sbloccare il suo acount e inserisci il codice ricevuto via sms.",
    "Congratulazione! Lei ha vinto un iPhone!!! Per ricevere il premio inserire numero della carta e cvv per le spese di spedizione, offerta scade oggi!!",
    "Caro studente la tua casella email sara disattivata. Per favore verifica la tua password subito al link sotto o perderai tutti i tuoi dati.",
]
GRUPPO_B = [
    "Buongiorno, le scriviamo per informarla che, a seguito di un aggiornamento dei sistemi di sicurezza, è necessario confermare l'accesso al suo conto entro 24 ore. Per completare la verifica le verrà chiesto di inserire il codice ricevuto via SMS. Grazie per la collaborazione.",
    "Ciao Giulia, sono Marco, il coach. Per la convocazione di sabato la federazione richiede una copia del documento e il codice fiscale: mandameli su WhatsApp a questo numero, il mio vecchio non funziona più. Non parlarne nel gruppo, è una questione riservata tra me e i giocatori convocati.",
    "Gentile amministrazione, come anticipato nella nostra ultima conversazione, vi comunichiamo il nuovo IBAN per il pagamento delle fatture di settembre. Vi chiediamo di procedere entro venerdì e di gestire la pratica con la dovuta riservatezza.",
]

## 2. Le regole

Due dizionari: a ogni nome di segnale corrisponde una lista di espressioni da cercare nel testo, in minuscolo.

- `testo.lower()`: il testo in minuscolo
- `any(e in testo for e in espressioni)`: vero se almeno un'espressione compare nel testo
- `testo.count("!")`: quante volte compare `!`
- `sum(c.isupper() for c in testo)`: quante lettere maiuscole ci sono

In [ ]:
REGOLE_FORMA = {
    "errori di ortografia frequenti": ["sicureza", "acount", "e stato", "sara disattivata", "congratulazione"],
}

REGOLE_CONTENUTO = {
    "urgenza": ["entro 24 ore", "immediatamente", "subito", "ultimo avviso", "scade oggi", "entro venerdì"],
    "codici o password": ["codice ricevuto", "codice via sms", "password", "pin"],
    "dati della carta o documenti": ["numero della carta", "cvv", "documento", "codice fiscale"],
    "cambio di iban": ["nuovo iban", "nuove coordinate", "iban aggiornato"],
    "cambio di canale": ["whatsapp", "questo numero", "non rispondere a questa email"],
    "segretezza": ["riservat", "non dirlo", "non parlarne"],
    "premio inatteso": ["hai vinto", "ha vinto", "premio", "rimborso"],
}

def segnali_forma(testo):
    t = testo.lower()
    trovati = [nome for nome, espr in REGOLE_FORMA.items() if any(e in t for e in espr)]
    if testo.count("!") >= 3:
        trovati.append("troppi punti esclamativi")
    maiuscole = sum(c.isupper() for c in testo) / max(1, sum(c.isalpha() for c in testo))
    if maiuscole > 0.15:
        trovati.append("troppe maiuscole")
    return trovati

def segnali_contenuto(testo):
    t = testo.lower()
    return [nome for nome, espr in REGOLE_CONTENUTO.items() if any(e in t for e in espr)]

## 3. Applicazione ai due gruppi (esercizio 3)

In [ ]:
def esamina(gruppo, nome_gruppo):
    print("=" * 70)
    print(nome_gruppo)
    for i, testo in enumerate(gruppo, start=1):
        f, c = segnali_forma(testo), segnali_contenuto(testo)
        print(f"\nmessaggio {i}: {testo[:70]}...")
        print(f"   forma    : {f if f else 'nessun segnale'}")
        print(f"   contenuto: {c if c else 'nessun segnale'}")

esamina(GRUPPO_A, "GRUPPO A - messaggi vecchio stile")
esamina(GRUPPO_B, "GRUPPO B - messaggi con testo curato")

**Domande**

- Quante volte scattano le regole sulla forma nel gruppo A? E nel gruppo B?
- E le regole sul contenuto?
- Perché la correttezza del testo non è più un indizio di affidabilità?

Risposte:

## 4. Falsi allarmi (esercizio 4)

- Falso positivo (falso allarme): un messaggio legittimo segnalato come sospetto
- Falso negativo: un messaggio fraudolento non segnalato

Aggiungere due messaggi legittimi inventati e verificare se il controllo produce falsi allarmi.

In [ ]:
LEGITTIMI = [
    "Gentili famiglie, si ricorda che la consegna del modulo di autorizzazione per l'uscita didattica va fatta entro venerdì tramite il registro elettronico.",
    "Il tuo ordine è stato spedito e sarà consegnato domani. Puoi seguire la spedizione dall'app del negozio.",
    # aggiungere qui altri messaggi legittimi
]
for testo in LEGITTIMI:
    print(testo[:70], "...")
    print("   contenuto:", segnali_contenuto(testo) or "nessun segnale")

**Domande**

- Quale regola ha prodotto un falso allarme? Come la modifichereste?
- Perché un segnale isolato (per esempio "entro venerdì") non basta, mentre la combinazione di più segnali (urgenza + codici, cambio di IBAN + segretezza) è molto più indicativa?

Proposta di modifica:

## 5. Approfondimento: un punteggio

Invece di un sì o no, si può assegnare un peso a ogni segnale e sommare i pesi. Modificare i pesi e la soglia e osservare l'effetto sui tre gruppi di messaggi.

In [ ]:
PESI = {"urgenza": 1, "codici o password": 3, "dati della carta o documenti": 3, "cambio di iban": 3,
        "cambio di canale": 2, "segretezza": 2, "premio inatteso": 2}
SOGLIA = 3

def punteggio(testo):
    return sum(PESI[s] for s in segnali_contenuto(testo))

for nome, gruppo in [("A", GRUPPO_A), ("B", GRUPPO_B), ("legittimi", LEGITTIMI)]:
    for testo in gruppo:
        p = punteggio(testo)
        print(f"{nome:10} punteggio {p:2}  {'SOSPETTO' if p >= SOGLIA else 'ok':8}  {testo[:55]}...")